In [2]:
# ablation_fixed.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model, clone_model
from tensorflow.keras.layers import LSTM, Dense, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping
import os
import random
import json
import pickle

# ==================== SET FULL DETERMINISM ====================
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# ========== OPSI DETERMINISME TINGGI (CPU ONLY) ==========
use_high_determinism = True
if use_high_determinism:
    try:
        tf.config.set_visible_devices([], 'GPU')
    except Exception:
        pass
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    print("⚠️ Mode determinisme tinggi diaktifkan - Menggunakan CPU hanya")

# ==================== KONFIGURASI ====================
ticker = "AAPL"
lookback = 5
epsilon = 1e-6

base_path = "saved_models_lstm"
os.makedirs(base_path, exist_ok=True)

# **IMPORTANT**: ini harus sama dengan finetuned_dir di kode utama
finetuned_models_dir = os.path.join(base_path, "finetuned_per_period")
os.makedirs(finetuned_models_dir, exist_ok=True)

sentiment_folder = "saved_sentiment_scraping"
os.makedirs(sentiment_folder, exist_ok=True)

best_model_path = os.path.join(base_path, f"best_lstm_model_{ticker.lower()}.keras")
best_hps_path = os.path.join(base_path, f"best_hyperparams_{ticker.lower()}.json")

# Folder untuk hasil ablation (CSV + plots)
ablation_results_path = os.path.join(base_path, "lstm_ablation_results")
os.makedirs(ablation_results_path, exist_ok=True)

# Jika True -> coba load fine-tuned model + scaler dari kode utama (tidak retrain)
use_saved_finetuned = True

# PENTING: pastikan tanggal sama persis dengan yang dipakai di kode utama
covid_periods = {
    'Sebelum COVID': ('2017-04-09', '2020-02-29'),   # <-- 29 Feb
    'Selama COVID':   ('2020-03-01', '2022-12-31'),
    'Setelah COVID':  ('2023-01-01', '2025-09-15')
}

# ==================== FUNGSI BANTU ====================
def mape(y_true, y_pred):
    y = np.maximum(np.abs(y_true), epsilon)
    return np.mean(np.abs((y_true - y_pred) / y)) * 100

def evaluate(y_true, y_pred):
    return {
        'MSE': mean_squared_error(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE': mape(y_true, y_pred)
    }

def print_metrics(m):
    for k, v in m.items():
        suf = '%' if k == 'MAPE' else ''
        print(f"{k}: {v:.4f}{suf}")

def create_dataset(arr):
    # arr expected shape (n_samples, n_features+1) with last column = target
    return arr[:, :-1], arr[:, -1]

# ==================== MEMUAT DATA YANG SUDAH ADA ====================
processed_features_path = os.path.join(base_path, f"processed_features_{ticker.lower()}.csv")
if not os.path.exists(processed_features_path):
    raise FileNotFoundError(f"processed_features not found: {processed_features_path}")

data_daily = pd.read_csv(processed_features_path, parse_dates=['Date'], index_col='Date')
sentiment_path = os.path.join(sentiment_folder, f"sentiment_{ticker.lower()}.csv")
if not os.path.exists(sentiment_path):
    raise FileNotFoundError(f"sentiment file not found: {sentiment_path}")

sentiment_df = pd.read_csv(sentiment_path, parse_dates=['Date'], index_col='Date')
data_daily['Sentiment'] = sentiment_df['Sentiment'].reindex(data_daily.index).fillna(0)
data_daily['Sentiment_Lag1'] = data_daily['Sentiment'].shift(1).fillna(0)

# ==================== FITUR & TARGET ====================
features_with_sentiment = [
    'Sentiment_Lag1', 'MA_5', 'MA_20', 'Return_1D',
    'Volatility_5D', 'RSI_14', 'Momentum_5D',
    'MA_5_20_ratio', 'RSI_MA5'
] + [f'lag_{i}' for i in range(1, lookback+1)]

features_without_sentiment = [
    'MA_5', 'MA_20', 'Return_1D',
    'Volatility_5D', 'RSI_14', 'Momentum_5D',
    'MA_5_20_ratio', 'RSI_MA5'
] + [f'lag_{i}' for i in range(1, lookback+1)]

target = 'Close'

# ==================== MUAT BEST HPs & MODEL UTAMA ====================
if not os.path.exists(best_hps_path):
    raise FileNotFoundError(f"best_hyperparams json not found: {best_hps_path}")

with open(best_hps_path, 'r') as f:
    best_hps = json.load(f)

if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"best model file not found: {best_model_path}")

print("📥 Memuat model utama (global) dan hyperparams...")
model_with_sentiment = load_model(best_model_path)
print("✅ Model global dimuat.")

# ==================== ABLATION STUDY ====================
def build_model_from_hps(num_features, hps):
    m = Sequential()
    m.add(Reshape((1, num_features), input_shape=(num_features,)))
    m.add(LSTM(units=int(hps['lstm_units']), return_sequences=False, activation='tanh'))
    m.add(Dense(units=int(hps['dense_units']), activation='relu'))
    m.add(Dense(1, activation='linear'))
    return m

def process_ablation_period(period_name, start_date, end_date):
    print(f"\n--- Memproses periode: {period_name} ({start_date} -> {end_date}) ---")
    # slicing eksplisit agar boundary konsisten
    start_dt = pd.to_datetime(start_date)
    end_dt   = pd.to_datetime(end_date)
    dfp = data_daily[(data_daily.index >= start_dt) & (data_daily.index <= end_dt)].copy()

    if dfp.empty:
        print(f"⚠️ Periode {period_name} kosong, lewati.")
        return None

    # Persiapan data dengan sentimen
    scaler_sent = MinMaxScaler()
    scaled_sent = scaler_sent.fit_transform(dfp[features_with_sentiment + [target]].values)
    X_sent, y_sent = create_dataset(scaled_sent)

    # Persiapan data tanpa sentimen
    scaler_no_sent = MinMaxScaler()
    scaled_no_sent = scaler_no_sent.fit_transform(dfp[features_without_sentiment + [target]].values)
    X_no_sent, y_no_sent = create_dataset(scaled_no_sent)

    # Split
    split_idx = int(len(X_sent) * 0.8)
    if split_idx < 1:
        raise ValueError(f"Data periode {period_name} terlalu kecil untuk split: {len(X_sent)} samples")

    X_train_sent, X_test_sent = X_sent[:split_idx], X_sent[split_idx:]
    y_train_sent, y_test_sent = y_sent[:split_idx], y_sent[split_idx:]

    X_train_no_sent, X_test_no_sent = X_no_sent[:split_idx], X_no_sent[split_idx:]
    y_train_no_sent, y_test_no_sent = y_no_sent[:split_idx], y_no_sent[split_idx:]

    # ===== Dengan Sentimen: coba load fine-tuned model & scaler yang dibuat oleh kode utama =====
    finetuned_model_file = os.path.join(finetuned_models_dir, f"finetuned_with_sentiment_{period_name.replace(' ','_').lower()}.keras")
    finetuned_scaler_file = os.path.join(finetuned_models_dir, f"scaler_{period_name.replace(' ','_').lower()}.pkl")

    model_sent = None
    scaler_used_for_sent = None

    if use_saved_finetuned and os.path.exists(finetuned_model_file) and os.path.exists(finetuned_scaler_file):
        print(f"📥 Menemukan fine-tuned model & scaler untuk periode {period_name}. Memuat dari disk.")
        model_sent = load_model(finetuned_model_file)
        try:
            with open(finetuned_scaler_file, 'rb') as f:
                scaler_used_for_sent = pickle.load(f)
        except Exception as e:
            print(f"⚠️ Gagal load scaler dari {finetuned_scaler_file}: {e}")
            scaler_used_for_sent = None
    else:
        print("⚠️ Fine-tuned model/scaler tidak tersedia untuk periode ini -> akan fine-tune dari global model (fallback).")
        # clone architecture & weights from global model
        try:
            model_sent = clone_model(model_with_sentiment)
            model_sent.set_weights(model_with_sentiment.get_weights())
        except Exception as e:
            print(f"⚠️ clone_model gagal: {e}. Membangun ulang arsitektur dari HP.")
            model_sent = build_model_from_hps(len(features_with_sentiment), best_hps)

        lr = float(best_hps.get('lr', 1e-3))
        opt = Adam(learning_rate=lr)
        model_sent.compile(optimizer=opt, loss=Huber())
        model_sent.fit(X_train_sent, y_train_sent,
                       validation_split=0.2,
                       epochs=100,
                       batch_size=16,
                       callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
                       verbose=0)
        # simpan agar di-run berikutnya ablation bisa langsung load
        try:
            model_sent.save(finetuned_model_file)
            # simpan scaler yang kita fit untuk periode ini
            with open(finetuned_scaler_file, 'wb') as f:
                pickle.dump(scaler_sent, f)
            print(f"💾 Model & scaler fine-tuned disimpan: {finetuned_model_file}, {finetuned_scaler_file}")
        except Exception as e:
            print(f"⚠️ Gagal menyimpan model/scaler fine-tuned: {e}")
        scaler_used_for_sent = scaler_sent

    # Jika kita berhasil load scaler dari file, gunakan itu untuk inverse_transform.
    # Jika tidak, gunakan scaler_sent yang kita fit di atas.
    if scaler_used_for_sent is None:
        scaler_used_for_sent = scaler_sent

    # Prediksi dan inverse transform untuk 'Dengan Sentimen'
    pred_scaled_sent = model_sent.predict(X_test_sent)
    pred_inv_sent = scaler_used_for_sent.inverse_transform(np.hstack([X_test_sent, pred_scaled_sent]))[:, -1]
    y_test_inv_sent = scaler_used_for_sent.inverse_transform(np.hstack([X_test_sent, y_test_sent.reshape(-1,1)]))[:, -1]

    # ===== Tanpa Sentimen (bangun & latih dari awal per-periode) =====
    print("🔧 Membangun & melatih model tanpa sentimen untuk periode ini.")
    model_no_sent = build_model_from_hps(len(features_without_sentiment), best_hps)
    lr = float(best_hps.get('lr', 1e-3))
    opt_no = Adam(learning_rate=lr)
    model_no_sent.compile(optimizer=opt_no, loss=Huber())
    model_no_sent.fit(X_train_no_sent, y_train_no_sent,
                      validation_split=0.2,
                      epochs=100,
                      batch_size=16,
                      callbacks=[EarlyStopping(patience=15, restore_best_weights=True)],
                      verbose=0)

    # Prediksi tanpa sentimen
    pred_scaled_no_sent = model_no_sent.predict(X_test_no_sent)
    pred_inv_no_sent = scaler_no_sent.inverse_transform(np.hstack([X_test_no_sent, pred_scaled_no_sent]))[:, -1]
    y_test_inv_no_sent = scaler_no_sent.inverse_transform(np.hstack([X_test_no_sent, y_test_no_sent.reshape(-1,1)]))[:, -1]

    # ========== Simpan hasil prediksi per-periode ke CSV & plot ==========
    dates_test = dfp.index[-len(y_test_inv_sent):]
    df_results = pd.DataFrame({
        'Date': pd.to_datetime(dates_test),
        'Actual': y_test_inv_sent,
        'Predicted_with_sentiment': pred_inv_sent,
        'Predicted_without_sentiment': pred_inv_no_sent
    })
    fname = period_name.replace(' ','_').lower()
    csv_path = os.path.join(ablation_results_path, f"ablation_{fname}.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"💾 Hasil prediksi disimpan: {csv_path}")

    # plot
    plt.figure(figsize=(12,6))
    plt.plot(dates_test, y_test_inv_sent, label='Actual', alpha=0.8)
    plt.plot(dates_test, pred_inv_sent, label='With Sentiment (fine-tuned)', linewidth=1.2)
    plt.plot(dates_test, pred_inv_no_sent, '--', label='Without Sentiment', linewidth=1.2)
    plt.title(f'LSTM Ablation: {period_name}')
    plt.legend(); plt.grid(True); plt.tight_layout()
    plot_path = os.path.join(ablation_results_path, f"ablation_plot_{fname}.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
    print(f"💾 Plot disimpan: {plot_path}")

    # metrics
    metrics_sent = evaluate(y_test_inv_sent, pred_inv_sent)
    metrics_no_sent = evaluate(y_test_inv_no_sent, pred_inv_no_sent)

    return {
        'dates': dates_test,
        'metrics_sent': metrics_sent,
        'metrics_no_sent': metrics_no_sent,
        'csv': csv_path,
        'plot': plot_path
    }

# ==================== RUN ABLATION ====================
def run_lstm_ablation():
    print("🚀 MEMULAI STUDI ABLASI LSTM")
    ablation_metrics = {}

    for period_name, (start_date, end_date) in covid_periods.items():
        res = process_ablation_period(period_name, start_date, end_date)
        if res is None:
            continue
        ablation_metrics[period_name] = {
            'with_sentiment': res['metrics_sent'],
            'without_sentiment': res['metrics_no_sent']
        }

    # simpan summary metrics
    metrics_json_path = os.path.join(ablation_results_path, 'ablation_metrics.json')
    with open(metrics_json_path, 'w') as f:
        json.dump(ablation_metrics, f, indent=4)
    print(f"\n💾 Semua metrik ablasi disimpan: {metrics_json_path}")

    # tampilkan ringkasan ke console (format rapi)
    print("\n📊 HASIL ABLASI LSTM (Perbandingan Dengan dan Tanpa Sentimen)")
    print("="*80)
    for period, metrics in ablation_metrics.items():
        print(f"\n=== {period} ===")
        print(f"{'METRIK':<12} {'Dengan Sentimen':<18} {'Tanpa Sentimen':<18} {'Perubahan':<12} {'Interpretasi':<20}")
        print("-"*80)
        for k in metrics['with_sentiment'].keys():
            v_sent = metrics['with_sentiment'][k]
            v_no = metrics['without_sentiment'][k]
            if k == 'MAPE':
                suf = '%'
                fmt_sent = f"{v_sent:.4f}{suf}"
                fmt_no = f"{v_no:.4f}{suf}"
            else:
                suf = ''
                fmt_sent = f"{v_sent:.4f}"
                fmt_no = f"{v_no:.4f}"

            # interpretasi perubahan
            if k == 'R2':
                change = ((v_sent - v_no) / (abs(v_no) + epsilon)) * 100
                interpretation = "↑ Lebih Baik" if change > 0 else ("↓ Lebih Buruk" if change < 0 else "≈ Sama")
            else:
                change = ((v_no - v_sent) / (abs(v_no) + epsilon)) * 100
                interpretation = "↑ Lebih Baik" if change > 0 else ("↓ Lebih Buruk" if change < 0 else "≈ Sama")
            change_str = f"{change:+.2f}%"
            print(f"{k:<12} {fmt_sent:<18} {fmt_no:<18} {change_str:<12} {interpretation:<20}")

    print("\n✅ Hasil ablasi LSTM disimpan di folder:", ablation_results_path)
    print("✨ Selesai.")

if __name__ == "__main__":
    run_lstm_ablation()


⚠️ Mode determinisme tinggi diaktifkan - Menggunakan CPU hanya
📥 Memuat model utama (global) dan hyperparams...
✅ Model global dimuat.
🚀 MEMULAI STUDI ABLASI LSTM

--- Memproses periode: Sebelum COVID (2017-04-09 -> 2020-02-29) ---
📥 Menemukan fine-tuned model & scaler untuk periode Sebelum COVID. Memuat dari disk.
1/7 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step

c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 16 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
🔧 Membangun & melatih model tanpa sentimen untuk periode ini.


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\keras\src\layers\reshaping\reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
💾 Hasil prediksi disimpan: saved_models_lstm\lstm_ablation_results\ablation_sebelum_covid.csv
💾 Plot disimpan: saved_models_lstm\lstm_ablation_results\ablation_plot_sebelum_covid.png

--- Memproses periode: Selama COVID (2020-03-01 -> 2022-12-31) ---
📥 Menemukan fine-tuned model & scaler untuk periode Selama COVID. Memuat dari disk.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
🔧 Membangun & melatih model tanpa sentimen untuk periode ini.


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\keras\src\layers\reshaping\reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
💾 Hasil prediksi disimpan: saved_models_lstm\lstm_ablation_results\ablation_selama_covid.csv
💾 Plot disimpan: saved_models_lstm\lstm_ablation_results\ablation_plot_selama_covid.png

--- Memproses periode: Setelah COVID (2023-01-01 -> 2025-09-15) ---
📥 Menemukan fine-tuned model & scaler untuk periode Setelah COVID. Memuat dari disk.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
🔧 Membangun & melatih model tanpa sentimen untuk periode ini.


c:\Users\andil\Downloads\prediksi_saham\venv\Lib\site-packages\keras\src\layers\reshaping\reshape.py:39: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
💾 Hasil prediksi disimpan: saved_models_lstm\lstm_ablation_results\ablation_setelah_covid.csv
💾 Plot disimpan: saved_models_lstm\lstm_ablation_results\ablation_plot_setelah_covid.png

💾 Semua metrik ablasi disimpan: saved_models_lstm\lstm_ablation_results\ablation_metrics.json

📊 HASIL ABLASI LSTM (Perbandingan Dengan dan Tanpa Sentimen)

=== Sebelum COVID ===
METRIK       Dengan Sentimen    Tanpa Sentimen     Perubahan    Interpretasi        
--------------------------------------------------------------------------------
MSE          0.2954             0.4346             +32.03%      ↑ Lebih Baik        
MAE          0.3570             0.4826             +26.03%      ↑ Lebih Baik        
R2           0.9969             0.9955             +0.15%       ↑ Lebih Baik        
MAPE         0.4943%            0.6850%            +27.84%      ↑ Lebih Baik        

=== Selama COVID ===
METRIK       Dengan Sentimen    Tanpa Sentimen     Perubahan    Interpr